# Day 5 — **Unit Test Generator from Notebook (V2)** — Colab + T4 GPU
Create pytest unit tests from a target Jupyter notebook (default: `/mnt/data/day5.ipynb`).  
Supports **Open‑Source** (HF Transformers; 4‑bit on T4) and **Frontier** (OpenAI/Anthropic) paths.  
Generates tests, builds a temp project, and **runs `pytest`** — all from a **Gradio UI**.

> ⚠️ Safety: Generated tests will execute your code. Review before running. Avoid notebooks with side effects.

## 0) Environment check

In [1]:
import os, sys, platform
print("Python:", sys.version)
print("Platform:", platform.platform())
try:
    import torch
    print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as e:
    print("Torch not installed yet:", e)

Python: 3.11.13 | packaged by conda-forge | (main, Jun  4 2025, 14:48:23) [GCC 13.3.0]
Platform: Linux-5.15.167.4-microsoft-standard-WSL2-x86_64-with-glibc2.39
Torch: 2.7.1 | CUDA: True
GPU: NVIDIA GeForce RTX 3060


## 1) Installs

In [2]:
# !pip install -q -U transformers accelerate bitsandbytes sentencepiece gradio pandas nbformat pytest
# !pip install -q -U openai==1.* anthropic==0.*

## 2) Imports & device

In [3]:
import os, re, io, json, textwrap, tempfile, subprocess, shutil, types, zipfile
from typing import List, Dict, Any, Optional, Tuple
import nbformat
import ast
import pandas as pd
import gradio as gr
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from dotenv import load_dotenv

# Optional frontier clients
try:
    from openai import OpenAI
except Exception:
    OpenAI = None
try:
    import anthropic
except Exception:
    anthropic = None

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

/home/hafnium/anaconda3/envs/llms/lib/python3.11/site-packages/sklearn/utils/_param_validation.py:14: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.3.3)
  from scipy.sparse import csr_matrix, issparse


## 3) (Optional) Google Drive mount for Colab

In [4]:
try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
    print("Drive mounted at /content/drive")
except Exception:
    print("Not running in Colab or Drive not available.")

Not running in Colab or Drive not available.


## 4) Auth & model defaults
- Set `OPENAI_API_KEY` and/or `ANTHROPIC_API_KEY` to enable the frontier path.
- Open‑source default: **`Qwen/Qwen2.5-Coder-7B-Instruct`** (fits T4 in 4‑bit).

In [5]:
# Load environment variables from .env file
load_dotenv(override=True)

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", None)
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", None)

openai_client = None
if OPENAI_API_KEY and OpenAI is not None:
    try:
        openai_client = OpenAI(api_key=OPENAI_API_KEY)
        print("✅ OpenAI client ready")
    except Exception as e:
        print("⚠️ OpenAI client init failed:", e)

anthropic_client = None
if ANTHROPIC_API_KEY and anthropic is not None:
    try:
        anthropic_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
        print("✅ Anthropic client ready")
    except Exception as e:
        print("⚠️ Anthropic client init failed:", e)

# --- Check if we can use advanced CUDA features ---
CUDA_ADVANCED = False
if torch.cuda.is_available():
    try:
        import bitsandbytes as bnb
        CUDA_ADVANCED = True
        print("✅ CUDA with quantization available")
    except Exception:
        print("⚠️ CUDA available but quantization disabled (missing CUDA toolkit)")

# --- Open‑source model defaults (CPU-friendly fallback) ---
if CUDA_ADVANCED:
    OS_DEFAULT = "Qwen/Qwen2.5-Coder-7B-Instruct"
else:
    OS_DEFAULT = "microsoft/Phi-3-mini-4k-instruct"  # Smaller, CPU-friendly
    print("🔄 Using CPU-optimized model as default")

OS_CHOICES = [
    "Qwen/Qwen2.5-Coder-7B-Instruct",
    "deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct",
    "HuggingFaceH4/zephyr-7b-beta",
    "microsoft/Phi-3-mini-4k-instruct",  # Best for CPU
    "microsoft/DialoGPT-medium",  # Fallback option
]

✅ OpenAI client ready
✅ Anthropic client ready
⚠️ CUDA available but quantization disabled (missing CUDA toolkit)
🔄 Using CPU-optimized model as default


/tmp/tmpxg5sw235/main.c:1:10: fatal error: cuda.h: No such file or directory
    1 | #include "cuda.h"
      |          ^~~~~~~~
compilation terminated.


## 5) Load open‑source model (4‑bit for T4)

In [6]:
_tok = None
_mdl = None

def load_os_model(model_id: str):
    global _tok, _mdl
    if _mdl is not None and getattr(_mdl, "name_or_path", None) == model_id:
        return _tok, _mdl
    print(f"Loading open-source model: {model_id}")
    
    # Start with basic kwargs
    kwargs = {}
    
    # Try to use 4-bit quantization if CUDA is available and properly configured
    if DEVICE == "cuda":
        try:
            # Test if BitsAndBytesConfig can be imported and used
            from transformers import BitsAndBytesConfig
            import bitsandbytes as bnb  # This will fail if CUDA headers aren't available
            
            kwargs["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_use_double_quant=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.bfloat16,
            )
            kwargs["device_map"] = "auto"
            kwargs["torch_dtype"] = torch.bfloat16
            print("✅ Using 4-bit quantization with CUDA")
            
        except Exception as e:
            print(f"⚠️ 4-bit quantization failed ({e}), falling back to standard CUDA loading")
            try:
                kwargs["device_map"] = "auto"
                kwargs["torch_dtype"] = torch.float16
                print("✅ Using standard CUDA loading")
            except Exception as e2:
                print(f"⚠️ CUDA loading failed ({e2}), falling back to CPU")
                kwargs = {}
    
    # Load tokenizer
    _tok = AutoTokenizer.from_pretrained(model_id, use_fast=True)
    if _tok.pad_token is None:
        _tok.pad_token = _tok.eos_token
    
    # Load model with fallback logic
    try:
        _mdl = AutoModelForCausalLM.from_pretrained(model_id, **kwargs)
        print(f"✅ Model loaded successfully on {_mdl.device if hasattr(_mdl, 'device') else 'unknown device'}")
    except Exception as e:
        print(f"⚠️ Model loading with {kwargs} failed: {e}")
        print("🔄 Trying CPU fallback...")
        _mdl = AutoModelForCausalLM.from_pretrained(model_id)
        print("✅ Model loaded on CPU")
    
    return _tok, _mdl

## 6) Notebook → module extractor (imports + defs + simple constants)

In [7]:
SIMPLE_ALLOWED_CONSTS = (ast.Constant,)
SIMPLE_ALLOWED_COMPOSITES = (ast.Tuple, ast.List, ast.Dict, ast.Set)

def is_simple_value(node: ast.AST) -> bool:
    if isinstance(node, SIMPLE_ALLOWED_CONSTS):
        return True
    if isinstance(node, SIMPLE_ALLOWED_COMPOSITES):
        if isinstance(node, (ast.Tuple, ast.List, ast.Set)):
            return all(is_simple_value(elt) for elt in node.elts)
        if isinstance(node, ast.Dict):
            return all(is_simple_value(k) and is_simple_value(v) for k, v in zip(node.keys, node.values))
    return False

def sanitize_cell_source(src: str) -> str:
    lines = []
    for line in src.splitlines():
        s = line.strip()
        if s.startswith("%") or s.startswith("!"):  # magics/shell
            continue
        if "get_ipython(" in line:
            continue
        lines.append(line)
    return "\n".join(lines)

def combine_code_from_notebook(nb_path: str) -> str:
    nb = nbformat.read(nb_path, as_version=4)
    chunks = []
    for cell in nb.cells:
        if cell.cell_type == "code":
            chunks.append(sanitize_cell_source(cell.source))
    return "\n\n".join(chunks)

def extract_module_parts(code: str) -> Tuple[str, Dict[str, str], Dict[str, str]]:
    """Return (module_code, functions_map, classes_map).
    - module_code: imports + simple constant assignments + defs (functions/classes); skips other top-level exec.
    """
    tree = ast.parse(code)
    imports = []
    assigns = []
    defs = []
    funcs, classes = {}, {}
    for node in tree.body:
        if isinstance(node, (ast.Import, ast.ImportFrom)):
            seg = ast.get_source_segment(code, node) or ""
            imports.append(seg)
        elif isinstance(node, ast.Assign) and all(isinstance(t, ast.Name) for t in node.targets) and is_simple_value(node.value):
            seg = ast.get_source_segment(code, node) or ""
            assigns.append(seg)
        elif isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            seg = ast.get_source_segment(code, node) or ""
            defs.append(seg)
            funcs[node.name] = seg
        elif isinstance(node, ast.ClassDef):
            seg = ast.get_source_segment(code, node) or ""
            defs.append(seg)
            classes[node.name] = seg
        else:
            # skip executable statements
            pass
    module_code = "\n\n".join(imports + assigns + defs)
    return module_code, funcs, classes

def scan_notebook(nb_path: str) -> Dict[str, Any]:
    code = combine_code_from_notebook(nb_path)
    module_code, funcs, classes = extract_module_parts(code)
    return {"module_code": module_code, "functions": funcs, "classes": classes}

## 7) Prompts for pytest generation

In [8]:
SYSTEM = (
    "You generate high-quality pytest unit tests. "
    "Return ONLY Python test code (no markdown or explanations). "
    "Assume the code under test is in 'module_under_test.py'. "
    "Import needed names like: from module_under_test import <names>. "
    "Rules: use pytest; no network/files; deterministic; cover typical, edge, and error cases; "
    "use pytest.raises for exceptions; keep runtime short."
)

def build_user_prompt(funcs: Dict[str,str], classes: Dict[str,str], extra_hints:str="") -> str:
    parts = ["Create pytest tests for the following code."]
    if funcs:
        parts.append("\n# Functions:")
        for n,s in funcs.items():
            parts.append(f"""\n## {n}\n{s}\n""")
    if classes:
        parts.append("\n# Classes:")
        for n,s in classes.items():
            parts.append(f"""\n## {n}\n{s}\n""")
    if extra_hints.strip():
        parts.append("\n# Additional requirements/hints:\n" + extra_hints.strip())
    parts.append(
        "\nConstraints:\n"
        "- Use only pytest + stdlib.\n"
        "- No I/O or network.\n"
        "- Import from module_under_test.\n"
        "- Keep tests readable and robust.\n"
    )
    return "\n".join(parts)

## 8) Generators (frontier + open‑source)

In [9]:
def gen_frontier(provider: str, model: str, prompt: str, temperature: float = 0.2, max_tokens: int = 1200) -> str:
    if provider == "OpenAI":
        if openai_client is None:
            raise RuntimeError("OpenAI client not configured")
        try:
            resp = openai_client.responses.create(
                model=model,
                input=[{"role":"system","content":SYSTEM},{"role":"user","content":prompt}],
                temperature=temperature,
                max_output_tokens=max_tokens,
            )
            text = resp.output_text
        except Exception:
            chat = openai_client.chat.completions.create(
                model=model,
                messages=[{"role":"system","content":SYSTEM},{"role":"user","content":prompt}],
                temperature=temperature,
                max_tokens=max_tokens,
            )
            text = chat.choices[0].message.content
        return text.strip()
    elif provider == "Anthropic":
        if anthropic_client is None:
            raise RuntimeError("Anthropic client not configured")
        msg = anthropic_client.messages.create(
            model=model, max_tokens=max_tokens, temperature=temperature,
            system=SYSTEM, messages=[{"role":"user","content":prompt}],
        )
        parts = []
        for blk in msg.content:
            if getattr(blk, "type", "") == "text":
                parts.append(blk.text)
        return "\n".join(parts).strip()
    else:
        raise ValueError("Unknown provider")

def gen_open_source(model_id: str, prompt: str, temperature: float = 0.2, top_p: float = 0.95, max_new_tokens: int = 1200) -> str:
    tok, mdl = load_os_model(model_id)
    messages = [{"role":"system","content":SYSTEM},{"role":"user","content":prompt}]
    
    # Prepare input with better error handling
    try:
        input_ids = tok.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt")
    except Exception as e:
        print(f"⚠️ Chat template failed: {e}")
        # Fallback to simple concatenation
        full_prompt = f"{SYSTEM}\n\n{prompt}"
        input_ids = tok.encode(full_prompt, return_tensors="pt")
    
    # Handle device placement more carefully
    model_device = next(mdl.parameters()).device if hasattr(mdl, 'parameters') else torch.device('cpu')
    input_ids = input_ids.to(model_device)
    
    print(f"🔄 Generating with model on {model_device}, input shape: {input_ids.shape}")
    
    with torch.no_grad():
        try:
            out = mdl.generate(
                input_ids=input_ids,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                top_p=top_p,
                do_sample=True,
                pad_token_id=tok.eos_token_id,
            )
        except Exception as e:
            print(f"⚠️ Generation failed with advanced settings: {e}")
            # Fallback with simpler settings
            out = mdl.generate(
                input_ids=input_ids,
                max_new_tokens=min(max_new_tokens, 512),
                do_sample=False,  # Greedy decoding
                pad_token_id=tok.eos_token_id,
            )
    
    gen = out[0, input_ids.shape[1]:]
    text = tok.decode(gen, skip_special_tokens=True).strip()
    
    # Strip any accidental fences
    text = re.sub(r"^```(?:python)?\n|\n```$", "", text, flags=re.IGNORECASE).strip()
    return text

## 9) Project builder, runner, and zipping

In [10]:
def write_project(module_code: str, tests_code: str) -> Tuple[str, str, str]:
    tmp = tempfile.mkdtemp()
    mod_path = os.path.join(tmp, "module_under_test.py")
    tst_path = os.path.join(tmp, "test_generated.py")
    with open(mod_path, "w", encoding="utf-8") as f:
        f.write(module_code)
    with open(tst_path, "w", encoding="utf-8") as f:
        f.write(tests_code)
    return tmp, mod_path, tst_path

def run_pytest(project_dir: str, timeout: int = 180) -> Tuple[int, str]:
    cmd = ["pytest", "-q"]
    try:
        proc = subprocess.run(cmd, cwd=project_dir, capture_output=True, text=True, timeout=timeout)
        rc = proc.returncode
        out = proc.stdout + "\n" + proc.stderr
        return rc, out
    except Exception as e:
        return 99, f"Error running pytest: {e}"

def zip_dir(dir_path: str, zip_path: str) -> str:
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for root, _, files in os.walk(dir_path):
            for fn in files:
                full = os.path.join(root, fn)
                rel = os.path.relpath(full, dir_path)
                zf.write(full, rel)
    return zip_path

## 10) Gradio App

In [11]:
DEFAULT_NB = "/mnt/data/day5.ipynb"
_cache = None  # holds last scan

def do_scan(nb_file, nb_path_text):
    # Choose uploaded file if present, else path text
    path = None
    if nb_file is not None:
        path = nb_file.name if hasattr(nb_file, "name") else nb_file
    else:
        path = nb_path_text.strip()
    if not path:
        return "⚠️ Provide a .ipynb via upload or path.", [], [], ""
    try:
        res = scan_notebook(path)
    except Exception as e:
        return f"⚠️ Parse error: {e}", [], [], ""
    func_names = list(res["functions"].keys())
    cls_names = list(res["classes"].keys())
    preview = (res["module_code"][:1500] + " ...") if len(res["module_code"]) > 1500 else res["module_code"]
    global _cache
    _cache = res
    
    # Return status, function choices, class choices, and preview
    return f"✅ Parsed: {path}", func_names, cls_names, preview

def do_generate(mode, frontier_model, os_model, funcs_sel, clss_sel, extra_hints, temperature, top_p):
    if not _cache:
        return "⚠️ Scan a notebook first.", None
    # Filter selections to only include valid choices (prevents validation errors)
    valid_funcs = funcs_sel or []
    valid_clss = clss_sel or []
    funcs = {k: _cache["functions"][k] for k in valid_funcs if k in _cache["functions"]}
    clss  = {k: _cache["classes"][k] for k in valid_clss if k in _cache["classes"]}
    if not funcs and not clss:
        return "⚠️ Select at least one function or class.", None
    prompt = build_user_prompt(funcs, clss, extra_hints)
    try:
        if mode == "Open‑Source (HF Transformers)":
            code = gen_open_source(os_model, prompt, temperature=temperature, top_p=top_p)
        elif mode == "Frontier (OpenAI)":
            code = gen_frontier("OpenAI", frontier_model, prompt, temperature=temperature)
        else:
            code = gen_frontier("Anthropic", frontier_model, prompt, temperature=temperature)
    except Exception as e:
        return f"⚠️ Generation error: {e}", None
    return code, None

def do_run_tests(tests_code):
    if not _cache:
        return "⚠️ Scan a notebook first.", None, None, None
    if not tests_code or not tests_code.strip():
        return "⚠️ Generate tests first.", None, None, None
    proj_dir, mod_path, tst_path = write_project(_cache["module_code"], tests_code)
    rc, log = run_pytest(proj_dir, timeout=240)
    # Save outputs for download
    res_path = os.path.join(proj_dir, "pytest_results.txt")
    with open(res_path, "w", encoding="utf-8") as f:
        f.write(log)
    zip_path = os.path.join(proj_dir, "test_project.zip")
    zip_dir(proj_dir, zip_path)
    summary = f"Exit code: {rc} — 0 means all tests passed"
    return summary, log, zip_path, res_path

with gr.Blocks(title="Unit Test Generator from Notebook (V2)") as app:
    gr.Markdown("## Unit Test Generator from Notebook (V2) — Day 5")
    with gr.Row():
        with gr.Column(scale=1):
            nb_file = gr.File(label="Upload .ipynb (optional)")
            nb_path_text = gr.Textbox(label="Or path to .ipynb", value=DEFAULT_NB)
            btn_scan = gr.Button("1) Scan Notebook", variant="primary")
            scan_status = gr.Textbox(label="Scan status")
            funcs = gr.CheckboxGroup(label="Functions", choices=[])
            clss  = gr.CheckboxGroup(label="Classes", choices=[])
            module_preview = gr.Textbox(label="Module preview (imports + defs + simple constants)", lines=14)
        with gr.Column(scale=1):
            mode = gr.Radio(
                label="Generation Path",
                value="Open‑Source (HF Transformers)",
                choices=["Open‑Source (HF Transformers)", "Frontier (OpenAI)", "Frontier (Anthropic)"]
            )
            frontier_model = gr.Textbox(label="Frontier model", value="gpt-4o-mini", placeholder="e.g., gpt-4o-mini / claude-3-5-sonnet-20240620")
            os_model = gr.Dropdown(label="Open‑source model", value=OS_DEFAULT, choices=OS_CHOICES)
            extra_hints = gr.Textbox(label="(Optional) Extra requirements for tests", lines=6, placeholder="e.g., focus on boundary conditions; parametrize common cases; check error messages...")
            temperature = gr.Slider(0.0, 1.5, value=0.2, step=0.05, label="Temperature (OS only)")
            top_p = gr.Slider(0.1, 1.0, value=0.95, step=0.05, label="top_p (OS only)")
            btn_gen = gr.Button("2) Generate Tests", variant="primary")
            tests_code = gr.Code(label="Generated pytest code")
        with gr.Column(scale=1):
            btn_run = gr.Button("3) Run Tests", variant="primary")
            summary = gr.Textbox(label="Pytest summary")
            log = gr.Textbox(label="Pytest log", lines=16)
            dl_zip = gr.File(label="Download test_project.zip")
            dl_res = gr.File(label="pytest_results.txt")

    # Update CheckboxGroup choices and clear selections when scanning
    def scan_and_update(*args):
        status, func_choices, cls_choices, preview = do_scan(*args)
        return (
            status,
            gr.CheckboxGroup(choices=func_choices, value=[]),  # Clear selections
            gr.CheckboxGroup(choices=cls_choices, value=[]),   # Clear selections  
            preview
        )
    
    btn_scan.click(fn=scan_and_update, inputs=[nb_file, nb_path_text], outputs=[scan_status, funcs, clss, module_preview])
    btn_gen.click(fn=do_generate, inputs=[mode, frontier_model, os_model, funcs, clss, extra_hints, temperature, top_p], outputs=[tests_code, summary])
    btn_run.click(fn=do_run_tests, inputs=[tests_code], outputs=[summary, log, dl_zip, dl_res])

print("✅ UI ready. In Colab/Jupyter, run:")
print("import gradio as gr; gr.close_all(); app.queue().launch(share=True)")

✅ UI ready. In Colab/Jupyter, run:
import gradio as gr; gr.close_all(); app.queue().launch(share=True)


### 11) Launch the app

In [12]:
# In Colab:
import gradio as gr
gr.close_all()
app.queue().launch(share=True)

* Running on local URL:  http://127.0.0.1:7862
* Running on public URL: https://482cbe3efe2fb0fe5c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Loading open-source model: microsoft/Phi-3-mini-4k-instruct
⚠️ 4-bit quantization failed (Command '['/usr/bin/gcc', '/tmp/tmpbpqohzrc/main.c', '-O3', '-shared', '-fPIC', '-Wno-psabi', '-o', '/tmp/tmpbpqohzrc/cuda_utils.cpython-311-x86_64-linux-gnu.so', '-lcuda', '-L/home/hafnium/anaconda3/envs/llms/lib/python3.11/site-packages/triton/backends/nvidia/lib', '-L/usr/lib/wsl/lib', '-I/home/hafnium/anaconda3/envs/llms/lib/python3.11/site-packages/triton/backends/nvidia/include', '-I/tmp/tmpbpqohzrc', '-I/home/hafnium/anaconda3/envs/llms/include/python3.11', '-I/home/hafnium/anaconda3/envs/llms/targets/x86_64-linux/include']' returned non-zero exit status 1.), falling back to standard CUDA loading
✅ Using standard CUDA loading


/tmp/tmpbpqohzrc/main.c:1:10: fatal error: cuda.h: No such file or directory
    1 | #include "cuda.h"
      |          ^~~~~~~~
compilation terminated.
`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


✅ Model loaded successfully on cuda:0
🔄 Generating with model on cuda:0, input shape: torch.Size([1, 263])


---
## Tips
- Keep the target notebook self‑contained. The extractor only includes **imports + simple constant assignments + defs**.
- If functions rely on complex top‑level state, factor that state into the functions or classes.
- If VRAM is tight on T4, switch to `microsoft/Phi-3-mini-4k-instruct` before generation.
- You can download a **zip** of the ephemeral project (module, tests, results) for inspection.